<a href="https://colab.research.google.com/github/juliaduboisas/plnEp1/blob/main/TF_IDF_%2B_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/juliaduboisas/plnEp1.git
%cd plnEp1

Cloning into 'plnEp1'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 40 (delta 14), reused 23 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 7.46 MiB | 9.85 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/plnEp1


In [2]:
import os

print("=== Estrutura de Arquivos ===")
for root, dirs, files in os.walk('.'):
    if '.git' in root:
        continue
    level = root.replace('.', '').count(os.sep)
    indent = ' ' * 4 * (level)
    print('{}{}/'.format(indent, os.path.basename(root)))
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print('{}{}'.format(subindent, f))

=== Estrutura de Arquivos ===
./
    .gitignore
    baselineModel.ipynb
    bertimbauModel.ipynb
    dataExploration.ipynb
    useModel.ipynb
    ep1-enunciado.pdf
    README.md
    tfidf_sublinear.ipynb
    roBERTaModel.ipynb
    TF_IDF_Word_+_Linear_SVM.ipynb
    ../data/train.csv
    sbertModel.ipynb
    pipelines/
        pipelineRegLogTFIDF.pkl
        pipelineDummy.pkl


In [3]:
import pandas as pd

# Carregar o dataset de treinamento
df = pd.read_csv('../data/train.csv')

# Exibir as informações gerais do DataFrame (tipos de dados e valores nulos)
print(df.info())
print("\n" + "="*50 + "\n")

# Exibir as primeiras 5 linhas para entender a estrutura dos dados
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20092 entries, 0 to 20091
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   resp_text  20092 non-null  object
 1   clarity    20092 non-null  object
dtypes: object(2)
memory usage: 314.1+ KB
None




,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

# 1. Separar features e target
X = df['resp_text']
y = df['clarity']

# 2. Converter os rótulos ('c1', 'c5', etc.) em números (0, 1, 2...)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Dividir os dados em treino e teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 4. Construir o Pipeline (TF-IDF + XGBoost)
# Limitamos a 5000 features para treinar mais rápido inicialmente
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('xgb', XGBClassifier(random_state=42, eval_metric='mlogloss'))
])

# 5. Treinar o modelo
print("Treinando o modelo TF-IDF + XGBoost... (isso pode levar alguns instantes)")
pipeline.fit(X_train, y_train)

# 6. Avaliar o modelo
print("Avaliando o modelo...")
y_pred = pipeline.predict(X_test)

print("\n=== Acurácia ===")
print(round(accuracy_score(y_test, y_pred), 4))

print("\n=== Relatório de Classificação ===")
target_names = le.inverse_transform(sorted(list(set(y_encoded))))
print(classification_report(y_test, y_pred, target_names=target_names))

Treinando o modelo TF-IDF + XGBoost... (isso pode levar alguns instantes)
Avaliando o modelo...

=== Acurácia ===
0.4277

=== Relatório de Classificação ===
              precision    recall  f1-score   support

          c1       0.44      0.40      0.42      1269
        c234       0.39      0.34      0.36      1371
          c5       0.45      0.54      0.49      1379

    accuracy                           0.43      4019
   macro avg       0.43      0.43      0.42      4019
weighted avg       0.42      0.43      0.42      4019



In [6]:
from sklearn.model_selection import GridSearchCV

# Definir a grade de hiperparâmetros para testar
# Usamos o prefixo 'xgb__' porque é o nome que demos ao passo do XGBoost no Pipeline
param_grid = {
    'xgb__max_depth': [3, 6],           # Profundidade máxima de cada árvore
    'xgb__learning_rate': [0.1, 0.2],   # Taxa de aprendizado
    'xgb__n_estimators': [100, 200]     # Número de árvores (iterações)
}

# Configurar o GridSearchCV
# cv=3 significa validação cruzada em 3 partes (ajuda a evitar overfitting)
# n_jobs=-1 usa todos os processadores disponíveis para acelerar a busca
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    verbose=2,
    n_jobs=-1
)

print("Iniciando o GridSearch... (Isso pode demorar alguns minutos!)\n")
grid_search.fit(X_train, y_train)

print("\n" + "="*50)
print("=== Melhores Hiperparâmetros Encontrados ===")
print(grid_search.best_params_)
print(f"Melhor Acurácia no Treino (CV): {grid_search.best_score_:.4f}")

# Avaliar o melhor modelo encontrado no conjunto de teste
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("\n=== Acurácia no Teste com o Melhor Modelo ===")
print(round(accuracy_score(y_test, y_pred_best), 4))

print("\n=== Relatório de Classificação (Melhor Modelo) ===")
print(classification_report(y_test, y_pred_best, target_names=target_names))

Iniciando o GridSearch... (Isso pode demorar alguns minutos!)

Fitting 3 folds for each of 8 candidates, totalling 24 fits

=== Melhores Hiperparâmetros Encontrados ===
{'xgb__learning_rate': 0.1, 'xgb__max_depth': 6, 'xgb__n_estimators': 200}
Melhor Acurácia no Treino (CV): 0.4467

=== Acurácia no Teste com o Melhor Modelo ===
0.4292

=== Relatório de Classificação (Melhor Modelo) ===
              precision    recall  f1-score   support

          c1       0.45      0.40      0.42      1269
        c234       0.38      0.32      0.35      1371
          c5       0.45      0.57      0.50      1379

    accuracy                           0.43      4019
   macro avg       0.43      0.43      0.42      4019
weighted avg       0.43      0.43      0.42      4019



In [7]:
# 4. Atualizar o Pipeline (TF-IDF com bigramas + XGBoost otimizado)
# ngram_range=(1, 2) para incluir unigramas e bigramas
# max_features=10000 para aumentar o vocabulário considerado
pipeline_ngram = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('xgb', XGBClassifier(
        random_state=42,
        eval_metric='mlogloss',
        learning_rate=0.1,
        max_depth=6,
        n_estimators=200
    ))
])

# 5. Treinar o novo modelo
print("Treinando o novo modelo TF-IDF (com bigramas) + XGBoost...")
pipeline_ngram.fit(X_train, y_train)

# 6. Avaliar o novo modelo
print("Avaliando o modelo atualizado...")
y_pred_ngram = pipeline_ngram.predict(X_test)

print("\n=== Nova Acurácia (com Bigramas e 10k features) ===")
print(round(accuracy_score(y_test, y_pred_ngram), 4))

print("\n=== Novo Relatório de Classificação ===")
print(classification_report(y_test, y_pred_ngram, target_names=target_names))


Treinando o novo modelo TF-IDF (com bigramas) + XGBoost...
Avaliando o modelo atualizado...

=== Nova Acurácia (com Bigramas e 10k features) ===
0.4362

=== Novo Relatório de Classificação ===
              precision    recall  f1-score   support

          c1       0.46      0.41      0.43      1269
        c234       0.39      0.33      0.36      1371
          c5       0.45      0.56      0.50      1379

    accuracy                           0.44      4019
   macro avg       0.43      0.44      0.43      4019
weighted avg       0.43      0.44      0.43      4019

